# Pennsylvania Severe Weather Atlas
## Notebook 02 — Study Scope and Historical Data Preparation

### Purpose

Notebook 1 inspected a 2025 sample. We will now define the study period and geographic scope, then prepare a historical Pennsylvania dataset.

### Notebook goals

1. Review NOAA coverage and reporting changes to select appropriate study years.
2. Define the statewide and Lehigh Valley geographic scope.
3. Prepare historical records for Tornado, Hail, and Thunderstorm Wind.
4. Validate and save the dataset with its source-file information.

### Imports and environment check

This notebook includes its own imports so it can run independently of Notebook 1.

The next cell loads our tools and identifies the Python environment. We will use `hashlib` later to create file fingerprints that help track the exact source files used.

In [1]:
from pathlib import Path
import sys
import hashlib

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import requests

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Virtual environment:", sys.prefix != sys.base_prefix)
print("Notebook 2 imports loaded successfully.")

Python: 3.13.5
Interpreter: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas\.venv\Scripts\python.exe
Virtual environment: True
Notebook 2 imports loaded successfully.


### Environment check — results

Notebook 2 is running Python 3.13.5 through the project's `.venv` interpreter.

The virtual-environment check returned `True`, and all imports loaded successfully.

## Loading the project configuration and saved inputs

This notebook needs its own project paths and DataFrames so it can run independently.

We will load the NOAA file inventory and source registry saved in Notebook 1.

Using `parse_dates` tells pandas to interpret `file_created` as dates, allowing us to sort and compare file versions.

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_PATHS = {
    "raw_noaa": PROJECT_ROOT / "data" / "raw" / "noaa_storm_events",
    "processed": PROJECT_ROOT / "data" / "processed",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "tables": PROJECT_ROOT / "reports" / "tables",
}

NOAA_DIRECTORY_URL = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"

inventory_path = PROJECT_PATHS["tables"] / "noaa_details_file_inventory.csv"
registry_path = PROJECT_PATHS["tables"] / "data_source_registry.csv"

file_inventory = pd.read_csv(
    inventory_path,
    parse_dates=["file_created"],
)
source_registry = pd.read_csv(registry_path)

print("Project root:", PROJECT_ROOT)
print("Inventory rows:", len(file_inventory))
print(
    "Inventory year range:",
    file_inventory["data_year"].min(),
    "to",
    file_inventory["data_year"].max(),
)

source_registry[["source_id", "status"]]

Project root: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas
Inventory rows: 77
Inventory year range: 1950 to 2026


,source_id,status
0,noaa_storm_events,2025 sample inspected; full audit pending


### Saved inputs — results

Notebook 2 resolved the project root correctly and loaded 77 inventory entries, with data-year labels ranging from 1950 to 2026.

The source registry retains the status:
`2025 sample inspected; full audit pending`.

## Working study period: 2011–2025

We will begin with 15 completed calendar years.

Starting in 2011 places the study after the 2010 change in the severe-thunderstorm hail-warning threshold from 0.75 inches to 1 inch. It also places the study after the Enhanced Fujita scale's introduction in 2007.

This choice avoids spanning those transitions. Reporting differences and record completeness still require assessment.

The next cell selects the newest listed file for each study year. Sorting by creation date and keeping the last entry identifies that version. We will then check for missing year entries.

In [3]:
STUDY_START_YEAR = 2011
STUDY_END_YEAR = 2025
STUDY_YEARS = list(range(STUDY_START_YEAR, STUDY_END_YEAR + 1))

STUDY_STATE = "PENNSYLVANIA"
FOCUS_EVENT_TYPES = ["Thunderstorm Wind", "Hail", "Tornado"]

study_inventory = (
    file_inventory.loc[file_inventory["data_year"].isin(STUDY_YEARS)]
    .sort_values(["data_year", "file_created"])
    .drop_duplicates(subset="data_year", keep="last")
    .reset_index(drop=True)
)

missing_years = sorted(
    set(STUDY_YEARS) - set(study_inventory["data_year"])
)

print("Study period:", STUDY_START_YEAR, "to", STUDY_END_YEAR)
print("Expected years:", len(STUDY_YEARS))
print("Annual files selected:", len(study_inventory))
print("Missing year entries:", missing_years)

study_inventory[["data_year", "file_created", "filename"]]

Study period: 2011 to 2025
Expected years: 15
Annual files selected: 15
Missing year entries: []


,data_year,file_created,filename
0,2011,2026-03-23,StormEvents_details-ftp_v1.0_d2011_c20260323.c...
1,2012,2026-03-23,StormEvents_details-ftp_v1.0_d2012_c20260323.c...
2,2013,2026-03-23,StormEvents_details-ftp_v1.0_d2013_c20260323.c...
3,2014,2026-03-23,StormEvents_details-ftp_v1.0_d2014_c20260323.c...
4,2015,2026-03-23,StormEvents_details-ftp_v1.0_d2015_c20260323.c...
5,2016,2026-03-23,StormEvents_details-ftp_v1.0_d2016_c20260323.c...
6,2017,2026-05-19,StormEvents_details-ftp_v1.0_d2017_c20260519.c...
7,2018,2026-03-23,StormEvents_details-ftp_v1.0_d2018_c20260323.c...
8,2019,2026-03-23,StormEvents_details-ftp_v1.0_d2019_c20260323.c...
9,2020,2026-03-23,StormEvents_details-ftp_v1.0_d2020_c20260323.c...


### Study-year selection — results

The selected inventory contains one file for every year from 2011 through 2025, with no gaps in the year sequence.

Each filename retains NOAA's file-creation date, identifying the selected version.

Record completeness will be assessed after loading the files.

## Geographic scope

The main analysis will cover Pennsylvania. Our Lehigh Valley view will include Lehigh and Northampton counties.

A county FIPS identifier combines a two-digit state code with a three-digit county code. Pennsylvania's state code is `42`.

We store these codes as text to preserve leading zeros, such as the `077` identifying Lehigh County.

This lookup table will help connect event records to county boundaries for mapping.

In [5]:
STUDY_STATE_FIPS = "42"

local_counties = pd.DataFrame({
    "county_name": ["LEHIGH", "NORTHAMPTON"],
    "state_fips": [STUDY_STATE_FIPS, STUDY_STATE_FIPS],
    "county_code": ["077", "095"],
})

local_counties["county_fips"] = (
    local_counties["state_fips"] + local_counties["county_code"]
)

LEHIGH_VALLEY_FIPS = local_counties["county_fips"].tolist()

print("Statewide scope:", STUDY_STATE)
print("Local view: Lehigh Valley")

local_counties

Statewide scope: PENNSYLVANIA
Local view: Lehigh Valley


,county_name,state_fips,county_code,county_fips
0,LEHIGH,42,077,42077
1,NORTHAMPTON,42,095,42095


### Geographic scope — results

The statewide scope is Pennsylvania. The local Lehigh Valley lookup contains:

| County | County FIPS |
|---|---|
| Lehigh | `42077` |
| Northampton | `42095` |

The three-digit county codes retain their leading zeros and combine correctly with Pennsylvania's state code, `42`.

## Downloading the historical source files

We will retrieve the 15 selected annual files and reuse files already present in the raw-data folder.

For each file, we will record its year, filename, source URL, size, and SHA-256 fingerprint. The fingerprint helps identify whether a file's contents change.

New downloads will be written to a temporary `.part` file before receiving their final filename.

We will save this information in a download manifest: a table documenting the source files used. Reading and auditing their records comes next.

In [6]:
download_records = []

for item in study_inventory.itertuples(index=False):
    raw_path = PROJECT_PATHS["raw_noaa"] / item.filename
    source_url = NOAA_DIRECTORY_URL + item.filename

    if raw_path.exists():
        status = "reused"
    else:
        print(f"{item.data_year}: downloading...", flush=True)
        response = requests.get(source_url, timeout=(15, 120))
        response.raise_for_status()

        if not response.content.startswith(b"\x1f\x8b"):
            raise ValueError(f"Unexpected file format for {item.data_year}.")

        temporary_path = raw_path.with_name(raw_path.name + ".part")
        temporary_path.write_bytes(response.content)
        temporary_path.replace(raw_path)
        status = "downloaded"

    download_records.append({
        "data_year": item.data_year,
        "filename": item.filename,
        "file_created": item.file_created,
        "source_url": source_url,
        "file_size_bytes": raw_path.stat().st_size,
        "sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
        "checked_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "status": status,
    })

    print(f"{item.data_year}: {status}", flush=True)

download_manifest = pd.DataFrame(download_records)

download_manifest_path = PROJECT_PATHS["tables"] / (
    f"noaa_download_manifest_{STUDY_START_YEAR}_{STUDY_END_YEAR}.csv"
)
download_manifest.to_csv(download_manifest_path, index=False)

print("Manifest saved to:", download_manifest_path)

download_manifest[["data_year", "status", "file_size_bytes"]]

2011: downloading...
2011: downloaded
2012: downloading...
2012: downloaded
2013: downloading...
2013: downloaded
2014: downloading...
2014: downloaded
2015: downloading...
2015: downloaded
2016: downloading...
2016: downloaded
2017: downloading...
2017: downloaded
2018: downloading...
2018: downloaded
2019: downloading...
2019: downloaded
2020: downloading...
2020: downloaded
2021: downloading...
2021: downloaded
2022: downloading...
2022: downloaded
2023: downloading...
2023: downloaded
2024: downloading...
2024: downloaded
2025: reused
Manifest saved to: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas\reports\tables\noaa_download_manifest_2011_2025.csv


,data_year,status,file_size_bytes
0,2011,downloaded,15695066
1,2012,downloaded,11820671
2,2013,downloaded,11725909
3,2014,downloaded,11084754
4,2015,downloaded,10041550
5,2016,downloaded,9065715
6,2017,downloaded,9340854
7,2018,downloaded,9585982
8,2019,downloaded,11602648
9,2020,downloaded,10444606


### Historical download — results

The download run accounted for all 15 selected annual files:
- 14 files were downloaded for 2011–2024.
- The existing 2025 file was reused.
- Every listed file has a nonzero size.

The download manifest was saved to:
`reports/tables/noaa_download_manifest_2011_2025.csv`

It records filenames, source URLs, file-creation dates, sizes, SHA-256 fingerprints, and check timestamps.

## Building the historical Pennsylvania subset

We will read one annual file at a time, select Pennsylvania records, and retain our three focus event types.

Each retained record will receive its source year and filename so we can trace it back to the original file.

Only the filtered records will be combined, keeping memory use manageable.

The annual summary will show file dimensions, Pennsylvania record counts, and focus-category counts. These are initial ingestion results; detailed quality checks follow.

In [7]:
focus_frames = []
ingestion_records = []

for item in download_manifest.itertuples(index=False):
    raw_path = PROJECT_PATHS["raw_noaa"] / item.filename

    annual_data = pd.read_csv(
        raw_path,
        compression="gzip",
        low_memory=False,
    )

    pa_mask = annual_data["STATE"].eq(STUDY_STATE)
    focus_mask = pa_mask & annual_data["EVENT_TYPE"].isin(FOCUS_EVENT_TYPES)

    year_focus = annual_data.loc[focus_mask].copy()
    year_focus["source_year"] = item.data_year
    year_focus["source_file"] = item.filename

    focus_frames.append(year_focus)

    ingestion_records.append({
        "year": item.data_year,
        "file_records": len(annual_data),
        "file_columns": annual_data.shape[1],
        "pa_records": int(pa_mask.sum()),
        "focus_records": len(year_focus),
    })

pa_focus_events = pd.concat(focus_frames, ignore_index=True)

ingestion_summary = pd.DataFrame(ingestion_records).sort_values("year")

print("Annual files read:", len(ingestion_summary))
print("Combined focus records:", f"{len(pa_focus_events):,}")

ingestion_summary

Annual files read: 15
Combined focus records: 15,399


,year,file_records,file_columns,pa_records,focus_records
0,2011,79091,51,1925,970
1,2012,64503,51,1685,1064
2,2013,59986,51,1803,992
3,2014,59475,51,1676,732
4,2015,57907,51,1470,720
5,2016,56005,51,898,485
6,2017,57041,51,1499,953
7,2018,62699,51,1556,632
8,2019,67864,51,2395,1571
9,2020,61281,51,1978,1513


### Historical data ingestion results

All **15 annual files covering 2011–2025** loaded successfully. Each source file contained **51 columns**.

Filtering for Pennsylvania’s Thunderstorm Wind, Hail, and Tornado records produced **15,399 reported event records**. The annual focus counts sum to the combined dataset’s row count.

- **Highest annual focus count:** 2019, with 1,571 records.
- **Lowest annual focus count:** 2016, with 485 records.
- **2025 cross-check:** 1,165 records, matching Notebook 01.

The added `source_year` and `source_file` columns connect each retained record to its original annual file.

These totals describe reported event records, not necessarily distinct physical storms. Loading and combining the files is complete; event-ID and date checks are next.

## Checking event IDs and start dates

We will check for missing or repeated event IDs, unreadable start dates, and start years that differ from their source file’s year.

Dates that cannot be parsed become `NaT`, pandas’ missing-date value. Repeated IDs are counted after their first occurrence.

In [8]:
pa_focus_events["begin_datetime"] = pd.to_datetime(
    pa_focus_events["BEGIN_DATE_TIME"],
    format="%d-%b-%y %H:%M:%S",
    errors="coerce",
)

event_ids = pa_focus_events["EVENT_ID"]
start_dates = pa_focus_events["begin_datetime"]

year_mismatch = (
    start_dates.notna()
    & start_dates.dt.year.ne(pa_focus_events["source_year"])
)

quality_summary = pd.Series({
    "Combined focus records": len(pa_focus_events),
    "Source years represented": pa_focus_events["source_year"].nunique(),
    "Missing event IDs": event_ids.isna().sum(),
    "Repeated non-missing event IDs": event_ids.dropna().duplicated().sum(),
    "Missing or unreadable start dates": start_dates.isna().sum(),
    "Start year differs from source year": year_mismatch.sum(),
}, name="count").to_frame()

print(
    "Record total matches ingestion summary:",
    len(pa_focus_events) == ingestion_summary["focus_records"].sum(),
)
print("Earliest recorded start:", start_dates.min())
print("Latest recorded start:", start_dates.max())

quality_summary

Record total matches ingestion summary: True
Earliest recorded start: 2011-02-27 21:43:00
Latest recorded start: 2025-12-19 14:50:00


,count
Combined focus records,15399
Source years represented,15
Missing event IDs,0
Repeated non-missing event IDs,0
Missing or unreadable start dates,0
Start year differs from source year,0


### Event-ID and date audit results

The combined dataset contains **15,399 records across 15 source years**, matching the ingestion summary.

Every record has an event ID, with **no repeated IDs**. All start dates parsed successfully, and every start year matches its source file’s year.

- **Earliest recorded start:** February 27, 2011, at 21:43.
- **Latest recorded start:** December 19, 2025, at 14:50.

No records were flagged by these ID and date checks.

## Checking geographic identifiers

NOAA’s `CZ_TYPE` identifies the geographic unit:

- `C`: County or parish.
- `Z`: Public forecast zone.
- `M`: Marine area.

The meaning of `CZ_FIPS` depends on that designation. We will check the geographic types, confirm Pennsylvania’s state code of 42, and look for missing or unreadable area codes before building county identifiers.

The cross-tabulation counts records by event type and geographic designation.

In [11]:
state_codes = pd.to_numeric(
    pa_focus_events["STATE_FIPS"], errors="coerce"
)
area_codes = pd.to_numeric(
    pa_focus_events["CZ_FIPS"], errors="coerce"
)
area_types = (
    pa_focus_events["CZ_TYPE"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

wrong_state = (
    state_codes.notna()
    & state_codes.ne(int(STUDY_STATE_FIPS))
)

print("Missing or unreadable state codes:", state_codes.isna().sum())
print("State codes other than Pennsylvania:", wrong_state.sum())
print("Missing or unreadable area codes:", area_codes.isna().sum())
print("Missing geographic types:", area_types.isna().sum())

geography_counts = pd.crosstab(
    pa_focus_events["EVENT_TYPE"],
    area_types.fillna("Missing"),
).reindex(FOCUS_EVENT_TYPES, fill_value=0)

geography_counts

Missing or unreadable state codes: 0
State codes other than Pennsylvania: 0
Missing or unreadable area codes: 0
Missing geographic types: 0


CZ_TYPE,C
EVENT_TYPE,
Thunderstorm Wind,12630
Hail,2441
Tornado,328


### Geographic audit results

All 15,399 focus records have the county designation (`CZ_TYPE = C`). No state codes, area codes, or geographic types were missing or unreadable, and every state code matched Pennsylvania.

The county-based records include:

- **Thunderstorm Wind:** 12,630 records.
- **Hail:** 2,441 records.
- **Tornado:** 328 records.

These results support using the county-code field to construct county identifiers.

## Building county identifiers and identifying Lehigh Valley records

A five-digit county FIPS identifier combines the two-digit state code with the three-digit county code. We store identifiers as text and use `str.zfill()` to supply leading zeros.

Our Lehigh Valley view includes:

- **Lehigh County:** 42077.
- **Northampton County:** 42095.

The new `is_lehigh_valley` column marks records belonging to either county with `True`, allowing us to select the local view from the statewide dataset.

In [12]:
pa_focus_events["county_fips"] = (
    state_codes.astype("Int64").astype("string").str.zfill(2)
    + area_codes.astype("Int64").astype("string").str.zfill(3)
)

pa_focus_events["is_lehigh_valley"] = (
    pa_focus_events["county_fips"].isin(LEHIGH_VALLEY_FIPS)
)

county_record_counts = pa_focus_events["county_fips"].value_counts()

local_count_summary = local_counties[
    ["county_name", "county_fips"]
].copy()

local_count_summary["record_count"] = (
    local_count_summary["county_fips"]
    .map(county_record_counts)
    .fillna(0)
    .astype(int)
)

print("Statewide focus records:", f"{len(pa_focus_events):,}")
print("Distinct county identifiers:", pa_focus_events["county_fips"].nunique())
print(
    "Lehigh Valley focus records:",
    f"{pa_focus_events['is_lehigh_valley'].sum():,}",
)

local_count_summary

Statewide focus records: 15,399
Distinct county identifiers: 67
Lehigh Valley focus records: 564


,county_name,county_fips,record_count
0,LEHIGH,42077,313
1,NORTHAMPTON,42095,251


### County identifiers and Lehigh Valley results

The statewide dataset contains **15,399 focus records with 67 distinct county identifiers**.

Our two-county Lehigh Valley subset contains **564 records**:

- **Lehigh County (42077):** 313 records.
- **Northampton County (42095):** 251 records.

The `is_lehigh_valley` flag identifies these local records within the statewide dataset.

## Auditing hail size and wind magnitude

Before interpreting severity, we will inspect the availability and range of the `MAGNITUDE` values.

Hail diameter is recorded in inches, while thunderstorm-wind speed is recorded in knots. Each event type therefore needs its own summary.

We will count missing or unreadable values, zeros, and negative values, then inspect the minimum, median, and maximum. These checks will help identify values that need closer examination.

In [13]:
magnitude_units = {
    "Hail": "inches",
    "Thunderstorm Wind": "knots",
}

magnitude_audit_records = []

for event_type, unit in magnitude_units.items():
    raw_values = pa_focus_events.loc[
        pa_focus_events["EVENT_TYPE"].eq(event_type),
        "MAGNITUDE",
    ]

    values = pd.to_numeric(raw_values, errors="coerce")

    magnitude_audit_records.append({
        "event_type": event_type,
        "unit": unit,
        "records": len(values),
        "missing_or_unreadable": int(values.isna().sum()),
        "zero_values": int(values.eq(0).sum()),
        "negative_values": int(values.lt(0).sum()),
        "minimum": values.min(),
        "median": values.median(),
        "maximum": values.max(),
    })

magnitude_audit = pd.DataFrame(magnitude_audit_records)

magnitude_audit

,event_type,unit,records,missing_or_unreadable,zero_values,negative_values,minimum,median,maximum
0,Hail,inches,2441,0,0,0,0.75,1.0,4.0
1,Thunderstorm Wind,knots,12630,0,0,0,25.00,50.0,104.0


### Magnitude audit results

All **2,441 hail records** and **12,630 thunderstorm-wind records** have readable magnitude values. Neither group contains missing, zero, or negative values.

| Event type | Unit | Minimum | Median | Maximum |
|---|---|---:|---:|---:|
| Hail | Inches | 0.75 | 1.00 | 4.00 |
| Thunderstorm Wind | Knots | 25 | 50 | 104 |

## Comparing magnitudes with severe thresholds

The numeric severe thresholds are **1 inch for hail diameter** and **50 knots for thunderstorm wind speed**.

Storm Data includes reports below these thresholds, so an event-type label alone does not establish that its magnitude meets the severe criterion.

We will count records below and at or above each threshold. These counts describe the reported magnitudes; they do not establish whether a particular record caused damage.

In [14]:
magnitude_thresholds = {
    "Hail": 1.0,
    "Thunderstorm Wind": 50.0,
}

threshold_audit_records = []

for event_type, threshold in magnitude_thresholds.items():
    values = pd.to_numeric(
        pa_focus_events.loc[
            pa_focus_events["EVENT_TYPE"].eq(event_type),
            "MAGNITUDE",
        ],
        errors="coerce",
    )

    threshold_audit_records.append({
        "event_type": event_type,
        "unit": magnitude_units[event_type],
        "threshold": threshold,
        "records": len(values),
        "below_threshold": int(values.lt(threshold).sum()),
        "at_or_above_threshold": int(values.ge(threshold).sum()),
    })

threshold_audit = pd.DataFrame(threshold_audit_records)

threshold_audit

,event_type,unit,threshold,records,below_threshold,at_or_above_threshold
0,Hail,inches,1.0,2441,809,1632
1,Thunderstorm Wind,knots,50.0,12630,54,12576


### Magnitude threshold results

- **Hail:** 1,632 of 2,441 records (66.9%) meet or exceed 1 inch. The remaining 809 records (33.1%) are below that threshold.
- **Thunderstorm Wind:** 12,576 of 12,630 records (99.6%) meet or exceed 50 knots. The remaining 54 records (0.4%) are below that threshold.

About one-third of the hail records fall below the severe-hail size threshold. Analyses specifically describing severe-sized hail will therefore need an explicit diameter filter.

## Distinguishing measured and estimated wind speeds

A recorded wind speed can represent an instrument measurement or an estimate. `MAGNITUDE_TYPE` identifies the reporting method and distinguishes gusts from sustained winds.

| Code | Meaning |
|---|---|
| MG | Measured gust |
| EG | Estimated gust |
| MS | Measured sustained wind |
| ES | Estimated sustained wind |

We will summarize these codes to understand how the wind magnitudes were obtained.

In [15]:
wind_type_labels = {
    "MG": "Measured gust",
    "EG": "Estimated gust",
    "MS": "Measured sustained wind",
    "ES": "Estimated sustained wind",
}

wind_types = (
    pa_focus_events.loc[
        pa_focus_events["EVENT_TYPE"].eq("Thunderstorm Wind"),
        "MAGNITUDE_TYPE",
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
    .fillna("Missing")
)

wind_type_summary = (
    wind_types.value_counts()
    .rename_axis("magnitude_type")
    .reset_index(name="record_count")
)

wind_type_summary["description"] = (
    wind_type_summary["magnitude_type"]
    .map(wind_type_labels)
    .fillna("Missing or unrecognized")
)

wind_type_summary["percent"] = (
    wind_type_summary["record_count"] / len(wind_types) * 100
).round(2)

wind_type_summary[
    ["magnitude_type", "description", "record_count", "percent"]
]

,magnitude_type,description,record_count,percent
0,EG,Estimated gust,12326,97.59
1,MG,Measured gust,304,2.41


### Wind reporting-method results

All 12,630 thunderstorm-wind records are coded as gusts:

- **Estimated gusts (EG):** 12,326 records — 97.59%.
- **Measured gusts (MG):** 304 records — 2.41%.

These two codes account for every wind record.

Wind-speed summaries therefore primarily describe estimated gusts. We should preserve the reporting-method field when comparing wind intensity.

## Auditing tornado intensity ratings

Tornado intensity is recorded in `TOR_F_SCALE`. Enhanced Fujita ratings use damage evidence to estimate wind intensity.

We will count the recorded rating labels and check for missing values. This establishes which categories are available for later tornado summaries.

In [16]:
tornado_ratings = (
    pa_focus_events.loc[
        pa_focus_events["EVENT_TYPE"].eq("Tornado"),
        "TOR_F_SCALE",
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

print("Tornado event records:", len(tornado_ratings))
print("Missing tornado ratings:", tornado_ratings.isna().sum())

tornado_rating_summary = (
    tornado_ratings.fillna("Missing")
    .value_counts()
    .sort_index()
    .rename_axis("tornado_rating")
    .reset_index(name="record_count")
)

tornado_rating_summary

Tornado event records: 328
Missing tornado ratings: 0


,tornado_rating,record_count
0,EF0,124
1,EF1,163
2,EF2,36
3,EF3,2
4,EFU,3


### Tornado rating audit results

All **328 tornado event records** have a populated rating field:

- **325 records** have ratings from EF0 through EF3.
- **3 records** are labeled EFU, indicating unknown intensity.

EF1 is the most frequent rating, with 163 records. The highest recorded rating is EF3, appearing in two records.

Zero missing fields does not mean every intensity is known: EFU explicitly identifies the three unrated records.

## Preparing analysis fields

We will add year and month fields for time-based summaries, plus two event-specific magnitude flags:

- `hail_ge_1in`: whether a hail record reports a diameter of at least 1 inch.
- `wind_ge_50kt`: whether a thunderstorm-wind record reports at least 50 knots.

For each flag, `True` means the reported magnitude meets the threshold, `False` means it falls below it, and `<NA>` means the flag does not apply to that event type.

In [17]:
pa_focus_events["event_year"] = pa_focus_events["begin_datetime"].dt.year
pa_focus_events["event_month"] = pa_focus_events["begin_datetime"].dt.month

magnitude_values = pd.to_numeric(
    pa_focus_events["MAGNITUDE"], errors="coerce"
).astype("Float64")

pa_focus_events["hail_ge_1in"] = (
    magnitude_values.ge(magnitude_thresholds["Hail"])
    .where(pa_focus_events["EVENT_TYPE"].eq("Hail"))
)

pa_focus_events["wind_ge_50kt"] = (
    magnitude_values.ge(magnitude_thresholds["Thunderstorm Wind"])
    .where(pa_focus_events["EVENT_TYPE"].eq("Thunderstorm Wind"))
)

flag_columns = ["hail_ge_1in", "wind_ge_50kt"]

flag_summary = pd.DataFrame({
    "true_records": pa_focus_events[flag_columns].eq(True).sum(),
    "false_records": pa_focus_events[flag_columns].eq(False).sum(),
    "not_applicable": pa_focus_events[flag_columns].isna().sum(),
})

flag_summary.index.name = "flag"

print("Prepared dataset shape:", pa_focus_events.shape)

flag_summary

Prepared dataset shape: (15399, 60)


,true_records,false_records,not_applicable
flag,,,
hail_ge_1in,1632,809,12958
wind_ge_50kt,12576,54,2769


### Analysis-field preparation results

The prepared dataset contains **15,399 records and 60 columns**.

- The hail flag identifies **1,632 records at or above 1 inch** and **809 below**.
- The wind flag identifies **12,576 records at or above 50 knots** and **54 below**.

For both flags, the true, false, and not-applicable counts sum to 15,399. Not-applicable values identify other event types.


## Saving the prepared dataset and audit tables

The historical dataset will be saved in `data/processed`, with its source-file information and analysis fields.

Supporting audit tables will be saved in `reports/tables`. The source registry will document that the 2011–2025 Pennsylvania focus subset has been prepared and its selected fields audited.


In [18]:
dataset_path = (
    PROJECT_PATHS["processed"] / "pa_focus_events_2011_2025.csv"
)
table_root = PROJECT_PATHS["tables"]

source_registry.loc[
    source_registry["source_id"].eq("noaa_storm_events"),
    "status",
] = "2011-2025 PA focus subset prepared; selected fields audited"

exports = {
    dataset_path: pa_focus_events,
    table_root / "pa_ingestion_audit_2011_2025.csv": ingestion_summary,
    table_root / "pa_id_date_audit_2011_2025.csv": (
        quality_summary.rename_axis("check").reset_index()
    ),
    table_root / "pa_geography_audit_2011_2025.csv": (
        geography_counts.reset_index()
    ),
    table_root / "lehigh_valley_focus_counts_2011_2025.csv": local_count_summary,
    table_root / "pa_magnitude_audit_2011_2025.csv": magnitude_audit,
    table_root / "pa_threshold_audit_2011_2025.csv": threshold_audit,
    table_root / "pa_wind_reporting_types_2011_2025.csv": wind_type_summary,
    table_root / "pa_tornado_ratings_2011_2025.csv": tornado_rating_summary,
    table_root / "pa_magnitude_flags_2011_2025.csv": flag_summary.reset_index(),
    registry_path: source_registry,
}

saved_outputs = []

for output_path, table in exports.items():
    table.to_csv(output_path, index=False)

    saved_outputs.append({
        "file": output_path.name,
        "rows": len(table),
        "columns": table.shape[1],
        "file_exists": output_path.is_file(),
    })

saved_output_summary = pd.DataFrame(saved_outputs)

print("Historical dataset saved to:", dataset_path)

saved_output_summary

Historical dataset saved to: c:\Users\rcolo\OneDrive\Desktop\Pennsylvania-Severe-Weather-Atlas\data\processed\pa_focus_events_2011_2025.csv


,file,rows,columns,file_exists
0,pa_focus_events_2011_2025.csv,15399,60,True
1,pa_ingestion_audit_2011_2025.csv,15,5,True
2,pa_id_date_audit_2011_2025.csv,6,2,True
3,pa_geography_audit_2011_2025.csv,3,2,True
4,lehigh_valley_focus_counts_2011_2025.csv,2,3,True
5,pa_magnitude_audit_2011_2025.csv,2,9,True
6,pa_threshold_audit_2011_2025.csv,2,6,True
7,pa_wind_reporting_types_2011_2025.csv,2,4,True
8,pa_tornado_ratings_2011_2025.csv,5,2,True
9,pa_magnitude_flags_2011_2025.csv,2,4,True


### Export results

The historical dataset was saved as `pa_focus_events_2011_2025.csv`, with **15,399 records and 60 columns**.

Nine supporting audit tables and the updated source registry were also saved. All 11 exported files were confirmed to exist.

## Verifying the saved dataset

CSV files do not preserve pandas data types. When reloading, we will explicitly restore county identifiers as text, the date column as datetime, and the flags as nullable booleans.

We will compare the saved row count, column names, and selected analysis fields with the in-memory dataset. Matching missing values will count as equal.

In [19]:
verified_events = pd.read_csv(
    dataset_path,
    dtype={
        "county_fips": "string",
        "is_lehigh_valley": "boolean",
        "hail_ge_1in": "boolean",
        "wind_ge_50kt": "boolean",
    },
    parse_dates=["begin_datetime"],
    low_memory=False,
)

roundtrip_checks = {
    "Row count matches": len(verified_events) == len(pa_focus_events),
    "Column names and order match": (
        verified_events.columns.tolist() == pa_focus_events.columns.tolist()
    ),
}

for column in [
    "EVENT_ID",
    "begin_datetime",
    "county_fips",
    "is_lehigh_valley",
    "hail_ge_1in",
    "wind_ge_50kt",
]:
    actual = verified_events[column]
    expected = pa_focus_events[column]

    same_value = actual.eq(expected).fillna(False)
    same_missing = actual.isna() & expected.isna()

    roundtrip_checks[f"{column} matches"] = bool(
        (same_value | same_missing).all()
    )

pd.Series(roundtrip_checks, name="passed").to_frame()

,passed
Row count matches,True
Column names and order match,True
EVENT_ID matches,True
begin_datetime matches,True
county_fips matches,True
is_lehigh_valley matches,True
hail_ge_1in matches,True
wind_ge_50kt matches,True


## Notebook 02 — Conclusion

Prepared and saved the **2011–2025 Pennsylvania focus dataset**, containing **15,399 reported event records and 60 columns**.

All eight reload checks passed. The saved dataset matched the in-memory table’s row count, column names and order, event IDs, start dates, county identifiers, local-view flag, and both magnitude-threshold flags.

The dataset contains **67 distinct county identifiers**. The Lehigh Valley subset contains **564 records**: 313 for Lehigh County and 251 for Northampton County.

### Interpretation points

- Counts represent reported event records, which may include segments of the same physical storm.
- **97.59% of wind records** are coded as estimated gusts.
- Hail records include sizes below 1 inch; the threshold flag supports analyses specifically examining severe-sized hail.
- Three tornado records have **unknown intensity (EFU)**.

The prepared dataset, source manifest, and audit tables provide the foundation for the next notebook.

**Next:** Explore annual and seasonal reporting patterns with statewide and Lehigh Valley visualizations.